# Matrix factorization Recommender System

Matrična faktorizacija (Matrix factorization) predstavlja način generisanja latentnih karakteristika (latent features) prilikom množenja dvije različite vrste entiteta.

# Inicijalno objašnjenje

Neka su:
- `U` - korisnici (kojima pravimo preporuke),
- `D` - predmeti (npr. filmovi koje su korisnici gledali),
- `R` - matrica korisničkih ocjena dimenzija `|U| x |D|`,
- `K` - broj latentnih karakteristika (hiperparametar),
- `P` - matrica korisničkih preferencija latentnih karakteristika dimenzija `|U| x K`
  koja nam govori koliko određeni korisnik preferira svaku od latentnih karakteristika,
- `Q` - matrica povezanosti predmeta sa latentnim karakteristika dimenzija `|D| x K`
  koja nam govori koliko određena latentna karakteristika odgovara svakom predmetu,

tada ćemo kreirati rekonstruisanu matricu korisničkih ocjena, sa predviđenim vrijednostima,
kao:

```math
R \approx P \times Q^T = \hat{R}
```

Predikcija za pojedinačni premdet se može dobiti kao:

```math
\hat{r}_{ij} = p_i^T q_j = \sum_{k=1}^{k} p_{ik} q_{kj}
```

Matrice `P` i `Q` se prvobitno inicijaliziraju pa dalje dobijaju metodom `SGD`.

Više o navedenom se može pročitati [ovdje](https://medium.com/data-science/recommendation-system-matrix-factorization-d61978660b4b).

# Implementacija

In [24]:
import numpy as np
np.random.seed(0)

def matrix_factorization(R, P, Q, K, steps=5000, alpha=0.0002, beta=0.02):
    """
    Args:
        R - rating matrix
        P - |U| * K (User features matrix)
        Q - |D| * K (Item features matrix)
        K - latent features
        steps - iterations
        alpha - learning rate
        beta - regularization parameter
    
    Returns:
        P - user features matrix
        Q - item features matrix
    """
    Q = Q.T

    for _ in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    # calculate error
                    eij = R[i][j] - np.dot(P[i,:],Q[:,j])

                    for k in range(K):
                        # calculate gradient with a and beta parameter
                        P[i][k] = P[i][k] + alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] = Q[k][j] + alpha * (2 * eij * P[i][k] - beta * Q[k][j])

        e = 0

        for i in range(len(R)):

            for j in range(len(R[i])):

                if R[i][j] > 0:

                    e = e + pow(R[i][j] - np.dot(P[i,:],Q[:,j]), 2)

                    for k in range(K):

                        e = e + (beta/2) * (pow(P[i][k],2) + pow(Q[k][j],2))
        # 0.001: local minimum
        if e < 0.001:

            break

    return P, Q.T

# Upotreba na MovieLens datasetu

In [25]:
import pandas as pd

# Sample dataset
# Zeros indicate missing ratings
R = [
    [5, 3, 0, 1],
    [4, 0, 0, 1],
    [1, 1, 0, 5],
    [1, 0, 0, 4],
    [0, 1, 5, 4],
]

# Initialize matrices P (dimensions |U| x K) and Q (dimensions |D| x K) with random values
num_users = len(R)
num_items = len(R[0])
K = 2  # Number of latent features
P = np.random.rand(num_users, K)
Q = np.random.rand(num_items, K)

nP, nQ = matrix_factorization(R, P, Q, K)
nR = np.dot(nP, nQ.T)

users = ['Korisnik 1', 'Korisnik 2', 'Korisnik 3', 'Korisnik 4', 'Korisnik 5']
movies = ['Film 1', 'Film 2', 'Film 3', 'Film 4']

# 1. Ispis Originalne matrice
print("Originalna sparse matrica:")
df_original = pd.DataFrame(R, index=users, columns=movies)
print(df_original)

# 2. Ispis Matrice nakon faktorizacije
print("\nMatrica nakon faktorizacije (Predikcije):")
# Zaokružujemo na 2 decimale radi preglednosti
df_predikcije = pd.DataFrame(np.round(nR, 2), index=users, columns=movies)
print(df_predikcije)


Originalna sparse matrica:
            Film 1  Film 2  Film 3  Film 4
Korisnik 1       5       3       0       1
Korisnik 2       4       0       0       1
Korisnik 3       1       1       0       5
Korisnik 4       1       0       0       4
Korisnik 5       0       1       5       4

Matrica nakon faktorizacije (Predikcije):
            Film 1  Film 2  Film 3  Film 4
Korisnik 1    5.07    2.71    5.61    1.00
Korisnik 2    3.91    2.09    4.44    1.00
Korisnik 3    1.13    0.60    3.40    4.97
Korisnik 4    0.93    0.50    2.75    3.98
Korisnik 5    2.96    1.58    4.83    4.02


# Analiza rezultata

Na osnovu poređenja **Originalne** i **Predviđene** matrice, zaključujemo sljedeće:

* **Preciznost:** Postojeće ocjene su gotovo identične (npr. **Korisnik 1 / Film 1** je ostao **5.07**). Model je uspješno naučio poznate profile.

* **Logika popunjavanja (Sličnost):**

    * **Korisnik 4 / Film 2 (0.5):** Sistem predviđa nisku ocjenu jer **Korisnik 4** ima sličan ukus kao **Korisnik 3** (obojica su Film 1 ocijenili niskom, a Film 4 visokom ocjenom). Pošto je Korisnik 3 "naglasio" loš kvalitet za Film 3 kroz latentne faktore, on se ne preporučuje ni Korisniku 4.

* **Zaključak:** Model je uspješno popunio praznine koristeći **kolaborativno filtriranje**  prepoznao je skrivene sličnosti u ocjenama i na osnovu njih predvidio ukus tamo gdje podaci nedostaju.